# URA Hackathon 2026 — Competition Data EDA
**Run on Kaggle** (attach the competition dataset) or Colab (upload CSV).

Pipeline: `Data Understanding → Preprocessing → EDA`

Palette `#e94b27 #6c761b #ffb8a7 #324712` · white bg · black text

> Figures auto-save to `figures/` in the working directory.

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────────────────────────
import subprocess, sys
try:
    from wordcloud import WordCloud
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "wordcloud", "-q"], check=True)
    from wordcloud import WordCloud

import os, re, glob, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Palette
PALETTE = ["#e94b27", "#6c761b", "#ffb8a7", "#324712"]
ORANGE, OLIVE, PEACH, GREEN = PALETTE

mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "text.color": "black",
    "axes.labelcolor": "black", "axes.titlecolor": "black",
    "xtick.color": "black", "ytick.color": "black",
    "axes.edgecolor": "black", "font.size": 12,
    "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110,
})
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=PALETTE)

base = "/kaggle/working" if os.path.isdir("/kaggle/working") else \
       ("/content" if os.path.isdir("/content") else ".")
FIG = Path(base) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print(f"Figures → {FIG}")

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")
    print(f"  saved {name}.png")

print("Setup complete.")

---
## 1. Data Understanding
Auto-detect the competition CSV, inspect shape, types, and missing values.

In [ ]:
# ── 1.1 Auto-detect competition train CSV ─────────────────────────────────────
REQUIRED_COLS = {"image_id", "ocr_text", "product_name"}

def find_train_csv():
    for root in ["/kaggle/input", "/content", "."]:
        if not os.path.isdir(root):
            continue
        for f in sorted(glob.glob(f"{root}/**/train_labels.csv", recursive=True)):
            try:
                peek = pd.read_csv(f, nrows=2, keep_default_na=False)
                if REQUIRED_COLS.issubset(peek.columns):
                    return f
            except Exception:
                continue
    return None

train_path = find_train_csv()
if train_path is None:
    raise FileNotFoundError(
        "Train CSV not found. Attach the competition dataset to this notebook.")

print(f"Found: {train_path}")
df_raw = pd.read_csv(train_path, keep_default_na=False, dtype=str)
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

In [ ]:
# ── 1.2 Column overview ───────────────────────────────────────────────────────
print("Columns and dtypes:")
print(df_raw.dtypes.to_string())
print()
df_raw.head(5)

In [ ]:
# ── 1.3 Missing value analysis ────────────────────────────────────────────────
null_count  = df_raw.isnull().sum()
empty_count = (df_raw == "").sum()
miss = pd.DataFrame({
    "null": null_count,
    "empty_str": empty_count,
    "total_missing": null_count + empty_count,
    "missing_%": ((null_count + empty_count) / len(df_raw) * 100).round(2)
})
print("Missing value summary:")
print(miss.to_string())

In [ ]:
# ── 1.4 Text-length statistics ────────────────────────────────────────────────
for col in ["ocr_text", "product_name"]:
    lens = df_raw[col].str.len()
    print(f"\n{col} character length:")
    print(lens.describe().round(1).to_string())

In [ ]:
# ── 1.5 Sample images (if available) ─────────────────────────────────────────
img_dirs = (glob.glob("/kaggle/input/**/images", recursive=True) +
            glob.glob("/kaggle/input/**/train_images", recursive=True) +
            glob.glob("/content/**/images", recursive=True))

if img_dirs:
    img_dir = img_dirs[0]
    img_files = sorted(glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png"))[:6]
    if img_files:
        fig, axes = plt.subplots(2, 3, figsize=(13, 8))
        id_col = df_raw.set_index("image_id")
        for ax, img_path in zip(axes.flat, img_files):
            img_id = Path(img_path).stem
            try:
                img = plt.imread(img_path)
                ax.imshow(img)
            except Exception:
                pass
            ax.axis("off")
            if img_id in id_col.index:
                prod = id_col.loc[img_id, "product_name"] or "(no label)"
                ax.set_title(prod[:35], fontsize=9, color=ORANGE if prod != "(no label)" else OLIVE)
        plt.suptitle("Sample Training Images — product label shown", fontsize=13, fontweight="bold")
        save(fig, "00_sample_images"); plt.show()
    else:
        print("Image files not found in", img_dir)
else:
    print("Images not attached to this session — skipping visual sample.")

---
## 2. Data Preprocessing
Normalize text, engineer features used in EDA and modeling.

In [ ]:
# ── 2.1 Text normalization ────────────────────────────────────────────────────
df = df_raw.copy()
df["ocr_text"]     = df["ocr_text"].str.strip().fillna("")
df["product_name"] = df["product_name"].str.strip().fillna("")

# ── 2.2 Binary label flags ────────────────────────────────────────────────────
df["has_product"] = (df["product_name"] != "").astype(int)
df["has_ocr"]     = (df["ocr_text"] != "").astype(int)

# ── 2.3 Text-length features ──────────────────────────────────────────────────
df["ocr_len"]            = df["ocr_text"].str.len()
df["ocr_word_count"]     = df["ocr_text"].apply(lambda s: len(s.split()) if s else 0)
df["product_token_count"] = df["product_name"].apply(lambda s: len(s.split()) if s else 0)

# ── 2.4 Diacritic density ─────────────────────────────────────────────────────
def diac_frac(s):
    if not s: return 0.0
    nfd = unicodedata.normalize("NFD", s)
    letters = sum(1 for c in nfd if c.isalpha())
    marks   = sum(1 for c in nfd if unicodedata.category(c) == "Mn")
    return marks / letters if letters else 0.0

df["diac_frac"] = df["ocr_text"].apply(diac_frac)

# ── 2.5 Product family key (diacritic-stripped, lowercase) ───────────────────
def normalize_key(s):
    if not s: return ""
    s = str(s).lower().replace("đ", "d")
    nfd = unicodedata.normalize("NFD", s)
    stripped = "".join(c for c in nfd if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9 ]", " ", stripped).strip()

df["product_key"] = df["product_name"].apply(normalize_key)

print(f"Preprocessed {len(df):,} rows")
print(f"  has_product : {df.has_product.sum():,}  ({df.has_product.mean():.1%})")
print(f"  has_ocr     : {df.has_ocr.sum():,}  ({df.has_ocr.mean():.1%})")
print(f"  mean ocr_len: {df.ocr_len.mean():.1f} chars")
print(f"  mean diac   : {df.diac_frac.mean():.3f}")

In [ ]:
# ── 2.6 Feature summary table ─────────────────────────────────────────────────
feat_cols = ["ocr_len", "ocr_word_count", "diac_frac",
             "product_token_count", "has_product", "has_ocr"]
df[feat_cols].describe().round(3)

---
## 3. Exploratory Data Analysis
Distributions, concentrations, correlations, and relationships between features and the target.

### 3.1 Product Label Distribution — Long-tail Concentration

In [ ]:
nz   = df[df.product_name != ""]["product_name"].value_counts()
top  = nz.head(15)[::-1]
cov  = top.sum() / nz.sum()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top)), top.values, color=ORANGE)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.index, fontsize=10)
ax.set_xlabel("Count in training set")
ax.set_title("Top-15 Product Labels — Long-tail Distribution")
ax.text(0.98, 0.04,
        f"Top-15 cover {cov:.0%} of {nz.sum():,} non-empty labels\n"
        f"{nz.size:,} distinct surface forms",
        transform=ax.transAxes, ha="right", color=GREEN, fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor=GREEN, alpha=0.85))
save(fig, "01_product_longtail"); plt.show()

### 3.2 Product Family Grouping — GT Fragmentation

In [ ]:
# Collapse all surface forms into canonical families
non_empty = df[df.product_name != ""].copy()
family = (
    non_empty.groupby("product_key")["product_name"]
    .agg(
        count="count",
        forms=lambda x: x.nunique(),
        example=lambda x: x.mode().iloc[0]
    )
    .sort_values("count", ascending=False)
    .head(15)
)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: family size by count
axes[0].barh(range(len(family)), family["count"], color=OLIVE)
axes[0].set_yticks(range(len(family)))
axes[0].set_yticklabels(family["example"], fontsize=9)
axes[0].set_xlabel("Total images (all surface forms)")
axes[0].set_title("Top-15 Product Families (collapsed)")
for i, v in enumerate(family["count"]):
    axes[0].text(v + 1, i, str(v), va="center", fontsize=9)

# Right: number of distinct surface forms (GT fragmentation)
colors_r = [ORANGE if v > 1 else GREEN for v in family["forms"]]
axes[1].barh(range(len(family)), family["forms"], color=colors_r)
axes[1].set_yticks(range(len(family)))
axes[1].set_yticklabels(family["example"], fontsize=9)
axes[1].set_xlabel("Distinct GT surface forms per family")
axes[1].set_title("GT Fragmentation — surface variants per family\n(orange = fragmented; hard F1 ceiling)")
for i, v in enumerate(family["forms"]):
    axes[1].text(v + 0.05, i, str(v), va="center", fontsize=9,
                 color=ORANGE if v > 1 else GREEN, fontweight="bold")

save(fig, "02_product_families"); plt.show()

### 3.3 Label Completeness — Empty Rate Analysis

In [ ]:
e_prod = (df.product_name == "").mean()
e_ocr  = (df.ocr_text == "").mean()
e_both = ((df.product_name == "") & (df.ocr_text == "")).mean()
e_prod_only = ((df.product_name == "") & (df.ocr_text != "")).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
vals = [e_prod, e_ocr, e_both]
lbls = ["product empty", "ocr empty", "both empty"]
axes[0].bar(lbls, vals, color=[ORANGE, OLIVE, GREEN])
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.008, f"{v:.1%}", ha="center", fontweight="bold")
axes[0].set_ylabel("Fraction of training set")
axes[0].set_ylim(0, max(vals) * 1.35)
axes[0].set_title("Empty Ground-Truth Rates")

# Stacked breakdown: what share of images have product / no product
groups = {
    "Has product": (df.product_name != "").mean(),
    "No product,\nhas OCR": e_prod_only,
    "No product,\nno OCR": e_both,
}
axes[1].bar(groups.keys(), groups.values(),
            color=[GREEN, PEACH, ORANGE])
for i, (k, v) in enumerate(groups.items()):
    axes[1].text(i, v + 0.008, f"{v:.1%}", ha="center", fontweight="bold")
axes[1].set_ylabel("Fraction of training set")
axes[1].set_ylim(0, max(groups.values()) * 1.35)
axes[1].set_title("Image Label Status Breakdown\n(motivates abstention strategy)")

save(fig, "03_empty_rate"); plt.show()

### 3.4 OCR Text Length Distribution

In [ ]:
L = df.ocr_len.values
cnt, edges = np.histogram(L, bins=50, range=(0, 530))

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(edges[:-1], cnt, width=np.diff(edges), align="edge",
       color=OLIVE, edgecolor=GREEN, linewidth=0.4)
ax.axvline(500, color=ORANGE, lw=2, ls="--", label="500-char GT truncation")
ax.axvline(L.mean(), color=GREEN, lw=1.5, ls=":",
           label=f"mean {L.mean():.0f}")
ax.set_xlabel("OCR text length (characters)")
ax.set_ylabel("Number of images")
ax.set_title(f"OCR Text Length — mean {L.mean():.0f}, median {np.median(L):.0f}")
ax.legend()

# Annotate empty-OCR bar
ax.text(0.01, 0.95, f"{(L==0).mean():.1%} images have no OCR text",
        transform=ax.transAxes, color=ORANGE, fontsize=10, va="top")

save(fig, "04_ocr_length"); plt.show()

### 3.5 OCR Text Length vs Product Label — Correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box plot: text length by product presence
grp_yes = df[df.has_product == 1]["ocr_len"]
grp_no  = df[df.has_product == 0]["ocr_len"]
axes[0].boxplot([grp_no, grp_yes],
                tick_labels=["No product label", "Has product label"],
                patch_artist=True,
                boxprops=dict(facecolor=PEACH),
                medianprops=dict(color=ORANGE, lw=2))
axes[0].set_ylabel("OCR text length (chars)")
axes[0].set_title("OCR Length vs Product Label Presence\n(images with more text tend to have product labels)")

# Word count vs product
axes[1].boxplot(
    [df[df.has_product == 0]["ocr_word_count"],
     df[df.has_product == 1]["ocr_word_count"]],
    tick_labels=["No product label", "Has product label"],
    patch_artist=True,
    boxprops=dict(facecolor=PEACH),
    medianprops=dict(color=GREEN, lw=2))
axes[1].set_ylabel("OCR word count")
axes[1].set_title("OCR Word Count vs Product Label Presence")

save(fig, "05_length_vs_product"); plt.show()

# Point-biserial correlation with has_product
from scipy.stats import pointbiserialr
print("\nPoint-biserial correlation with has_product:")
for feat in ["ocr_len", "ocr_word_count", "diac_frac"]:
    r, p = pointbiserialr(df["has_product"], df[feat])
    print(f"  {feat:<20} r={r:+.3f}  p={p:.2e}")


### 3.6 Vietnamese Diacritic Analysis — Why OCR Engine Matters

In [ ]:
# Diacritic density in training OCR text
text_bearing = df[df.ocr_text != ""]
dd = text_bearing["diac_frac"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of diacritic fraction
cnt, edges = np.histogram(dd, bins=30, range=(0, 0.6))
axes[0].bar(edges[:-1], cnt, width=np.diff(edges), align="edge",
            color=PEACH, edgecolor=ORANGE, linewidth=0.5)
axes[0].axvline(dd.mean(), color=GREEN, lw=2,
                label=f"mean {dd.mean():.2f}")
axes[0].set_xlabel("Diacritic marks per letter")
axes[0].set_ylabel("Images")
axes[0].set_title("Vietnamese Diacritic Density in OCR Text\n(~25% of letters carry diacritics)")
axes[0].legend()

# Right: top diacritic marks in GT product names — with readable Vietnamese names
all_products = " ".join(df[df.product_name != ""]["product_name"].values)
nfd = unicodedata.normalize("NFD", all_products)
diac_chars = [c for c in nfd if unicodedata.category(c) == "Mn"]
diac_freq = Counter(diac_chars).most_common(10)

# Map each Unicode combining mark to its Vietnamese tone/modifier name
DIAC_NAMES = {
    "́": "Acute ´\n(sắc = rising tone)",
    "̀": "Grave `\n(huyền = falling tone)",
    "̃": "Tilde ~\n(ngã = tumbling tone)",
    "̉": "Hook above\n(hỏi = asking tone)",
    "̣": "Dot below .\n(nặng = heavy tone)",
    "̂": "Circumflex ^\n(â / ê / ô vowel)",
    "̆": "Breve ˘\n(ă vowel)",
    "̛": "Horn\n(ơ / ư vowel)",
    "̈": "Diaeresis ¨\n(rare / loanword)",
}
dc_labels = [
    DIAC_NAMES.get(c, f"U+{ord(c):04X}")
    for c, _ in diac_freq
]
dc_counts = [cnt for _, cnt in diac_freq]

axes[1].bar(range(len(dc_labels)), dc_counts, color=ORANGE)
axes[1].set_xticks(range(len(dc_labels)))
axes[1].set_xticklabels(dc_labels, fontsize=8, ha="center")
axes[1].set_ylabel("Count in product labels")
axes[1].set_title("Top Diacritic Marks in Product Labels\n(lost entirely by PaddleOCR / RapidOCR → high CER)")
for i, v in enumerate(dc_counts):
    axes[1].text(i, v + 10, str(v), ha="center", fontsize=8)

save(fig, "06_diacritic_analysis"); plt.show()


### 3.7 Word Cloud of OCR Text

In [ ]:
# Strip diacritics for word-cloud rendering (avoids font issues)
def to_latin(s):
    s = str(s).lower().replace("đ", "d")
    nfd = unicodedata.normalize("NFD", s)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

# All OCR text concatenated
corpus_words = []
for txt in df[df.ocr_text != ""]["ocr_text"]:
    tokens = re.findall(r"[a-zA-ZđĐÀ-ỹ]+", txt)
    corpus_words.extend([to_latin(t) for t in tokens if len(t) >= 2])

corpus_str = " ".join(corpus_words)

# Common stopwords to exclude
VI_STOP = {"cua", "va", "hay", "trong", "cac", "cung", "thi", "la",
           "duoc", "co", "khong", "nay", "nha", "hang", "ban", "gia",
           "san", "pham", "mo", "tai", "voi", "sau", "cach", "them"}

wc = WordCloud(
    width=1200, height=600, background_color="white",
    colormap="RdYlGn", stopwords=VI_STOP,
    max_words=120, min_font_size=10,
    prefer_horizontal=0.8
).generate(corpus_str)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud — OCR Text Corpus (diacritics stripped for display)",
             fontsize=14, fontweight="bold")
save(fig, "07_wordcloud"); plt.show()

### 3.8 Top N-grams in OCR Text

In [ ]:
def get_ngrams(series, n, top_k=15):
    tokens_list = [re.findall(r"[a-zA-ZđĐÀ-ỹ]+", str(t).lower())
                   for t in series if t]
    ngrams = []
    for toks in tokens_list:
        for i in range(len(toks) - n + 1):
            ngrams.append(" ".join(toks[i:i+n]))
    return Counter(ngrams).most_common(top_k)

ocr_texts = df[df.ocr_text != ""]["ocr_text"]
uni = get_ngrams(ocr_texts, 1)
bi  = get_ngrams(ocr_texts, 2)
tri = get_ngrams(ocr_texts, 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, ngrams, title, color in [
    (axes[0], uni[::-1], "Top Unigrams", ORANGE),
    (axes[1], bi[::-1],  "Top Bigrams",  OLIVE),
    (axes[2], tri[::-1], "Top Trigrams", GREEN),
]:
    terms = [t for t, _ in ngrams]
    counts = [c for _, c in ngrams]
    ax.barh(range(len(terms)), counts, color=color)
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms, fontsize=9)
    ax.set_xlabel("Frequency")
    ax.set_title(title)

plt.suptitle("Most Frequent N-grams in OCR Text Corpus", fontsize=14, fontweight="bold")
save(fig, "08_ngrams"); plt.show()

### 3.9 Product Name Structure Analysis

In [ ]:
prod_texts = df[df.product_name != ""]["product_name"]
prod_tokens = get_ngrams(prod_texts, 1, top_k=20)
token_lengths = df[df.product_name != ""]["product_token_count"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: top product unigrams
terms = [t for t, _ in prod_tokens[::-1]]
counts = [c for _, c in prod_tokens[::-1]]
axes[0].barh(range(len(terms)), counts, color=ORANGE)
axes[0].set_yticks(range(len(terms)))
axes[0].set_yticklabels(terms, fontsize=9)
axes[0].set_xlabel("Frequency in product labels")
axes[0].set_title("Top Words in Product Names")

# Right: product name token-length distribution
tok_cnt = token_lengths.value_counts().sort_index()
axes[1].bar(tok_cnt.index.astype(str), tok_cnt.values, color=OLIVE)
axes[1].set_xlabel("Number of tokens in product name")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Product Name Length (tokens)\nmean={token_lengths.mean():.1f}")
for i, (k, v) in enumerate(tok_cnt.items()):
    axes[1].text(i, v + 2, str(v), ha="center", fontsize=9)

save(fig, "09_product_structure"); plt.show()

### 3.10 Feature Correlation with Product Label Presence

In [ ]:
from scipy.stats import pointbiserialr

feats = ["ocr_len", "ocr_word_count", "diac_frac", "has_ocr"]
corr_results = []
for feat in feats:
    r, p = pointbiserialr(df["has_product"], df[feat])
    corr_results.append((feat, r, p))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: point-biserial correlation
names_r = [c[0] for c in corr_results]
rs = [c[1] for c in corr_results]
colors_r = [ORANGE if r > 0 else GREEN for r in rs]
axes[0].barh(names_r, rs, color=colors_r)
axes[0].axvline(0, color="black", lw=1)
axes[0].set_xlabel("Point-biserial correlation with has_product")
axes[0].set_title("Feature Correlation with Product Label Presence")
for i, (r, (_, rv, p)) in enumerate(zip(rs, corr_results)):
    axes[0].text(r + (0.003 if r >= 0 else -0.003), i,
                 f"r={rv:+.3f}  p={p:.2e}",
                 va="center",
                 ha="left" if r >= 0 else "right",
                 fontsize=9)

# Heatmap: full Pearson correlation matrix of numeric features
num_feats = ["ocr_len", "ocr_word_count", "diac_frac", "has_product", "has_ocr"]
corr_mat = df[num_feats].corr()
im = axes[1].imshow(corr_mat, cmap="RdYlGn", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(num_feats)))
axes[1].set_yticks(range(len(num_feats)))
axes[1].set_xticklabels(num_feats, rotation=40, ha="right", fontsize=9)
axes[1].set_yticklabels(num_feats, fontsize=9)
for i in range(len(num_feats)):
    for j in range(len(num_feats)):
        axes[1].text(j, i, f"{corr_mat.iloc[i,j]:.2f}",
                     ha="center", va="center", fontsize=9,
                     color="black" if abs(corr_mat.iloc[i,j]) < 0.7 else "white")
plt.colorbar(im, ax=axes[1])
axes[1].set_title("Feature Correlation Matrix")

save(fig, "10_feature_correlations"); plt.show()

---
## EDA Summary

| Finding | Implication |
|:--|:--|
| Top-5 product families dominate | Rules tuned to dominant families capture most of the F1 gain |
| Same product appears under token-disjoint surface forms | Hard F1 ceiling — no rule-based method can fully resolve |
| ~40% of train images have no product label | Abstention is better than guessing for unclear images |
| ~25% of OCR letters carry diacritics | CER is diacritic-sensitive → diacritic-preserving OCR is required |
| OCR text length positively correlates with product presence | Longer text → more likely to contain brand/product info |
| Brand-name n-grams appear frequently in raw OCR | Simple pattern matching can recover most product labels |

> **Next:** `model_training.ipynb` — OCR engine fine-tuning, product extraction head (CalibratedRuleHead), and CV-gated experiment log.